# 05 — Graph transformer (15 min)

**Question:** does a graph model of the network around a train beat XGBoost with hand-made network features, and where does its gain come from?

1. Setup and training curves
2. Headline results
3. Paired comparisons: graph vs controls, refit, vs XGBoost, ensemble
4. Where the graph helps
5. Ensemble weight (sensitivity only)
6. Findings and decisions

**Model** (`swissdelay.models.graph_transformer`): Graphormer-style transformer.

- **Graph:** one subgraph per prediction point (`swissdelay.features.graph`). It contains the stops from the current station to the target plus their 1-hop neighbours in the static graph of planned runs, up to 32 nodes and 96 edges.
- **Features:**
  - nodes: events, mean delay and share late over 10 / 30 / 60 min, measured before the minute of *t* − 2 min;
  - edges: planned runs and running time, plus trains and delay gained over 10 / 30 min;
  - [CLS] token: the 66 tabular features of XGBoost `full_network_plus`.
- **Attention:** biased by hop distance and edge features.
- **Size and training:** 3 layers, d = 64, 4 heads, 293 k parameters. L1 loss, AdamW with a one-cycle learning-rate schedule, early stopping on April. The refit trains on Aug–Apr for the same 15 epochs.

**Variants:**

- `graph`: the full model;
- `nograph`: the [CLS] token only, i.e. a neural network on the tabular features;
- `random`: the same nodes, with edges rewired at random inside each subgraph;
- `graph_refit`: `graph` retrained on Aug–Apr;
- `avg`: a fixed 50/50 average of `graph_refit` and XGBoost `full_network_plus` (refit). The weight is chosen in advance, not tuned on May.

Validation month May 2026, common subset, 15-min horizon only. MAE in min of Δd, day-clustered 95 % CIs; `diff` negative = better.

In [ ]:
# 0. Setup: XGBoost and graph-transformer predictions on the same points, with breakdown columns
import json

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px

from swissdelay import config
from swissdelay.evaluation.metrics import evaluate, mae_diff_ci
from swissdelay.features import graph as gr
from swissdelay.viz import GRAY, SERIES, use_style

use_style()
pd.set_option("display.float_format", "{:,.3f}".format)
P, R = config.PROCESSED, config.REPORTS
KEYS = ["trip_key", "stop_seq", "horizon_min"]
VARIANTS = ["nograph", "graph", "random", "graph_refit"]

con = duckdb.connect()
gt = " ".join(f"JOIN read_parquet('{P}/gt_{v}_predictions_valid_h15.parquet') g_{v} USING ({', '.join(KEYS)})"
              for v in VARIANTS)
preds = con.sql(f"""
SELECT x.*, nw.pred_xgb_full, {", ".join(f"g_{v}.pred_gt_{v}" for v in VARIANTS)},
       f.hour, n.net_cur_n
FROM read_parquet('{P}/tabular_predictions_valid.parquet') x
JOIN read_parquet('{P}/tabular_predictions_valid_network.parquet') nw USING ({", ".join(KEYS)})
{gt}
JOIN read_parquet('{P}/features_2026-05.parquet') f USING ({", ".join(KEYS)})
LEFT JOIN read_parquet('{P}/network_2026-05.parquet') n USING ({", ".join(KEYS)})
WHERE x.horizon_min = 15
""").df()
for c in ("in_common_subset", "is_disruption_day"):
    preds[c] = preds[c].astype(bool)
preds["pred_avg"] = (preds["pred_gt_graph_refit"] + preds["pred_xgb_full_network_plus"]) / 2

# subgraph size of each point
g = gr.load(gr.GRAPH_DIR / "h15" / "2026-05")
sizes = g["points"][KEYS].assign(n_nodes=np.asarray(g["node_n"]), n_edges=np.asarray(g["edge_n"]))
preds = preds.merge(sizes, on=KEYS, how="left")
common = preds[preds["in_common_subset"]].copy()
print(f"{len(preds):,} points at 15 min, {len(common):,} in the common subset")

In [ ]:
# 1.1 Training curves: April MAE by epoch (early-stopping runs) and training loss (refit)
hist = []
for v in VARIANTS:
    info = json.loads((R / f"gt_{v}_fit_info_valid_h15.json").read_text())
    hist += [h | {"variant": v} for h in info["history"]]
hist = pd.DataFrame(hist)
display(hist.pivot(index="epoch", columns="variant", values="tune_mae").drop(columns="graph_refit", errors="ignore"))
fig = px.line(hist.melt(id_vars=["epoch", "variant"], value_vars=["train_l1", "tune_mae"]).dropna(),
              x="epoch", y="value", color="variant", line_dash="variable", markers=True,
              color_discrete_map={"nograph": GRAY, "graph": SERIES[0], "random": SERIES[1], "graph_refit": SERIES[2]},
              title="Training L1 (solid) and April MAE (dashed) by epoch, 15 min")
fig.add_hline(y=0.624, line_dash="dot", line_color=GRAY, annotation_text="XGBoost, April (no refit)")
fig.update_layout(yaxis_title="MAE (min)", legend_title=None)
fig.show()

> **Finding 1.1 (training).**
> - Every run converges. On April, `nograph` levels off at **0.650** (best epoch 12 of 14), `graph` at **0.621** and `random` at **0.622** (both still improving very slightly at epoch 15, the end of the learning-rate schedule).
> - XGBoost `full_network_plus` without refit reached 0.624 on April. **The neural network alone starts 0.026 behind the trees; the graph makes up the gap.**
> - Cost: about 1 min per epoch for `nograph`, and 4–4.5 min for `graph` and `random` on an M4 Pro (MPS), so about 1 h per run.

## 2. Headline results

In [ ]:
# 2.1 Validation, common subset, 15 min: MAE with CIs and difference vs the historical baseline
MODELS = ["pred_historical", "pred_xgb_full", "pred_xgb_norefit_full_network_plus", "pred_xgb_full_network_plus",
          "pred_gt_nograph", "pred_gt_random", "pred_gt_graph", "pred_gt_graph_refit", "pred_avg"]
res = evaluate(common, MODELS, reference="pred_historical")
res[["model", "n", "mae", "mae_lo", "mae_hi", "p90_abs_err", "diff_vs_ref", "diff_lo", "diff_hi"]]

> **Finding 2.1 (headline, common subset, 182,706 points).**
>
> | Model | MAE [95 % CI] | P90 abs. error |
> |---|---|---|
> | Historical median | 0.883 | 1.883 |
> | XGBoost `full` | 0.720 | 1.512 |
> | GT `nograph` | 0.734 | 1.548 |
> | XGBoost `full_network_plus`, no refit | 0.709 | 1.485 |
> | GT `random` | 0.708 | 1.502 |
> | GT `graph` | 0.705 | 1.499 |
> | XGBoost `full_network_plus` + refit (**deployed champion**) | 0.701 [0.679, 0.720] | 1.468 |
> | **GT `graph` + refit** | **0.697** [0.676, 0.716] | 1.480 |
> | **50/50 average (GT refit + XGBoost)** | **0.692** [0.671, 0.710] | **1.455** |
>
> - The GT refit has a **lower mean error but a heavier tail** than XGBoost: its P90 is 1.480, vs 1.468.
> - The average is best on both the mean and the tail.

## 3. Paired comparisons

In [ ]:
# 3.1 Paired MAE differences on the same points (day-clustered CI)
def paired(df, a, b, label, by=None):
    keys = ["horizon_min"] + ([by] if by else [])
    rows = []
    for k, grp in df.groupby(keys, observed=True):
        y, day = grp["delta_min"].to_numpy(), grp["operating_day"].to_numpy()
        ea, eb = np.abs(y - grp[a].to_numpy()), np.abs(y - grp[b].to_numpy())
        d, lo, hi = mae_diff_ci(day, ea, eb)
        rows.append(dict(zip(keys, k if isinstance(k, tuple) else (k,)))
                    | {"comparison": label, "n": len(grp), "mae_ref": eb.mean(), "diff": d, "lo": lo, "hi": hi,
                       "rel_%": 100 * d / eb.mean(), "significant": hi < 0 or lo > 0})
    return pd.DataFrame(rows)

PAIRS = [
    ("pred_gt_graph", "pred_gt_nograph", "graph vs nograph (value of the subgraph)"),
    ("pred_gt_random", "pred_gt_nograph", "random vs nograph (station information, wrong topology)"),
    ("pred_gt_graph", "pred_gt_random", "graph vs random (value of the real topology)"),
    ("pred_gt_graph_refit", "pred_gt_graph", "refit vs no refit (GT)"),
    ("pred_gt_graph", "pred_xgb_norefit_full_network_plus", "GT vs XGBoost, both without refit"),
    ("pred_gt_graph_refit", "pred_xgb_full_network_plus", "GT vs XGBoost, both refit"),
    ("pred_avg", "pred_xgb_full_network_plus", "average vs XGBoost"),
    ("pred_avg", "pred_gt_graph_refit", "average vs GT"),
]
comp = pd.concat([paired(common, a, b, label) for a, b, label in PAIRS])
comp[["comparison", "diff", "lo", "hi", "rel_%", "significant"]].round(4)

> **Finding 3.1 (paired differences, min).**
> - **The subgraph is worth −0.029 [−0.031, −0.028] to the same network** (`graph` vs `nograph`, −4.0 %). That is more than the 33 hand-made network features gave XGBoost (−0.019 vs `full`).
> - **Most of it comes from station-level information, not from the topology.** With randomly rewired edges the model still gets −0.027. The real topology adds −0.0025 [−0.0033, −0.0016]: significant, but about 9 % of the gain.
> - **Refit:** −0.0077 for the GT, the same as for XGBoost (−0.008).
> - **GT vs XGBoost on equal terms:** −0.0037 without refit, −0.0035 [−0.0051, −0.0020] with refit. **The graph transformer is the best single model, by 0.5 %.**
> - **The 50/50 average is best overall:** −0.0089 [−0.0099, −0.0080] vs XGBoost (−1.3 %) and −0.0054 vs the GT alone. The two models make different errors.

## 4. Where the graph helps

Paired difference by stratum for two comparisons:

- **`graph` vs `nograph`:** the value of the subgraph for the same network;
- **`graph_refit` vs XGBoost champion:** the GT against the deployed model.

`rel_%` is relative to the reference model's MAE in that stratum. `*` marks a significant difference.

In [ ]:
# 4.1 Strata
common["d0_bucket"] = pd.cut(common["d0_min"], [-99, 1, 3, 10, 999], right=False,
                             labels=["on time < 1", "minor 1-3", "moderate 3-10", "major > 10"])
common["traffic"] = pd.cut(common["net_cur_n"], [-1, 10, 30, 60, 1e9], labels=["≤ 10 events", "11-30", "31-60", "> 60"])
common["period"] = pd.cut(common["hour"], [-1, 5, 8, 15, 18, 24],
                          labels=["night 0-5", "morning peak 6-8", "day 9-15", "evening peak 16-18", "late 19-23"])
common["subgraph"] = pd.cut(common["n_nodes"], [0, 10, 20, 31, 32], labels=["≤ 10 nodes", "11-20", "21-31", "32 (cap)"])
STRATA = ["d0_bucket", "traffic", "period", "subgraph", "category", "is_disruption_day"]
for col in STRATA:
    print(col, common[col].value_counts(sort=False).to_dict())

In [ ]:
# 4.2 Relative gain by stratum (%, * = significant)
def where(a, b, by):
    r = paired(common, a, b, "", by=by)
    r["cell"] = r["rel_%"].map("{:+.1f}".format) + np.where(r["significant"], "*", "")
    return r.set_index(by)[["n", "mae_ref", "cell"]]

for col in STRATA:
    t = where("pred_gt_graph", "pred_gt_nograph", col).rename(columns={"cell": "graph vs nograph", "mae_ref": "mae_nograph"})
    t2 = where("pred_gt_graph_refit", "pred_xgb_full_network_plus", col)
    display(t.join(t2[["mae_ref", "cell"]].rename(columns={"mae_ref": "mae_xgb", "cell": "GT refit vs XGBoost"})))

In [ ]:
# 4.3 Chart: GT refit vs XGBoost by current delay and by subgraph size
r = pd.concat([paired(common, "pred_gt_graph_refit", "pred_xgb_full_network_plus", "", by=c)
                 .rename(columns={c: "stratum"}).assign(by=c) for c in ["d0_bucket", "subgraph"]])
r["stratum"] = r["stratum"].astype(str)
fig = px.bar(r, x="stratum", y="rel_%", facet_col="by", color="significant",
             color_discrete_map={True: SERIES[0], False: GRAY},
             title="GT refit vs XGBoost champion: relative MAE change by stratum (%)")
fig.update_xaxes(matches=None, title=None)
fig.update_layout(yaxis_title="Δ MAE (%)")
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("by=", "")))
fig.show()

> **Finding 4.1–4.3 (where the graph helps).**
> - **The subgraph helps the network in every stratum** (`graph` vs `nograph`, −2.4 to −5.4 %, all significant).
>   - It helps most for **late trains** (−5.1 % at 3–10 min late, −5.3 % above 10 min), at **night** (−5.4 %) and in **small subgraphs** (−4.9 % up to 10 nodes).
>   - It helps least at **busy stations** (more than 30 events in 30 min: −2.4 %) and in **subgraphs at the 32-node cap** (−2.7 %).
> - **Against XGBoost, the GT's small advantage is not uniform:**
>   - **better**, significant in most strata: EC −1.8 %, disruption day −1.2 % (one day), evening peak −1.0 %, stations with 11–30 events −1.0 %, IC −0.9 %;
>   - **worse at the big hubs:** +1.5 % where more than 60 events were measured in 30 min, and +0.5 % for subgraphs at the 32-node cap. RE is +0.3 % (not significant).
> - **Interpretation.** Around the big hubs the subgraph is truncated: neighbours are kept by traffic, up to 32 nodes. XGBoost's aggregate features summarise all events, so they don't suffer from this cap. A larger cap, or a hub-aware neighbour selection, is the obvious improvement. It also explains why averaging works: the two models are strong in different places.
> - Major delays (> 10 min): −2.3 % vs XGBoost, not significant (1,654 points).

## 5. Ensemble weight (sensitivity only)

The deployed comparison uses a fixed 50/50 weight chosen before looking at May. This table shows how sensitive the result is to that choice. Picking the best weight here would be tuning on the validation month, so it is **not** used to choose.

In [ ]:
# 5.1 MAE of w × GT refit + (1 − w) × XGBoost, common subset
y = common["delta_min"].to_numpy()
w_tab = pd.DataFrame({"w_gt": np.round(np.arange(0, 1.01, 0.1), 1)})
w_tab["mae"] = [np.mean(np.abs(w * common["pred_gt_graph_refit"] + (1 - w) * common["pred_xgb_full_network_plus"] - y))
                for w in w_tab["w_gt"]]
w_tab.set_index("w_gt").T

> **Finding 5.1 (ensemble weight, sensitivity only).**
> - The MAE is flat between a GT weight of 0.4 and 0.8 (0.692–0.693). Every weight from 0.2 to 0.9 beats both models alone (0.701 and 0.697).
> - The fixed 50/50 weight sits at the bottom of this curve, so choosing it in advance cost nothing.
> - No weight is tuned on May.

## 6. Findings and decisions

### Results (validation May 2026, common subset, 15 min)

| Model | MAE | P90 | vs XGBoost champion (paired) |
|---|---|---|---|
| Historical median | 0.883 | 1.883 | |
| XGBoost `full_network_plus` + refit (deployed) | 0.701 | 1.468 | reference |
| GT `nograph` (neural network, same features) | 0.734 | 1.548 | +0.033 |
| GT `graph` + refit | 0.697 | 1.480 | **−0.0035** [−0.005, −0.002] |
| 50/50 average | **0.692** | **1.455** | **−0.009** [−0.010, −0.008] (−1.3 %) |

### Answers

- **Does an explicit graph beat hand-made network features?** Yes, slightly.
  - On the same features and on equal terms (both refit), the graph transformer beats XGBoost by 0.0035 min (0.5 %).
  - It has to make up the usual 0.026–0.033 handicap of a neural network on tabular data first.
- **What does the graph add?** About 4 % for the same network (−0.029). **90 % of that comes from station-level information** (the recent state of each station on and around the route); only 10 % from the real topology (graph vs random −0.0025).
  - The hand-made features in XGBoost compress the stations into a few sums: they lose part of this detail, but not the topology.
- **Where?**
  - The graph helps most for late trains, at night and in small subgraphs.
  - The GT beats XGBoost on EC, IC, the evening peak and the disruption day.
  - It is worse at the busiest hubs, where the 32-node cap truncates the neighbourhood.
- **Best model:** the 50/50 average of the two, −1.3 % vs XGBoost, with a lighter tail too.

### Decisions

| # | Topic | Decision | Evidence |
|---|---|---|---|
| 1 | Deployed model | **XGBoost `full_network_plus` stays**, at all horizons. The GT (−0.5 %) and the average (−1.3 %) are below the 2 % promotion rule, and would add graph extraction and PyTorch to the nightly pipeline | 2.1, 3.1 |
| 2 | Graph transformer | Reported as a research result at 15 min. **30 and 60 min not run** | — |
| 3 | Sequence model | Skipped: previous stops add ≤ 0.003 (notebook 03) | notebook 03 |
| 4 | Possible follow-ups (not planned) | (a) station-level features around the route in XGBoost; (b) a larger or hub-aware node cap for the GT | 3.1, 4.2 |
| 5 | Next | Quantile XGBoost (P10 / P90), then the June test (used once) | — |